In [32]:
import pandas as pd
import json
import re
import os
from gzip import open as gzip_open
from chython.files import SDFRead, SDFWrite
import pickle


In [23]:
def process_json_file(file_path):
    """Обрабатывает один JSON-файл и возвращает DataFrame с аннотациями и преобразованными данными."""
    
    # Открытие JSON-файла и загрузка данных
    with open(file_path, 'r') as file:
        json_data = json.load(file)

    # Извлечение списка аннотаций из JSON
    annotations = json_data['Annotations']['Annotation']
    
    # Нормализация данных из аннотаций в таблицу (DataFrame), преобразование вложенных данных в плоскую структуру
    annotations_df = pd.json_normalize(
        annotations,
        record_path='Data',  # Распаковываем вложенные данные из поля 'Data'
        meta=[
            'SourceName', 'SourceID', 'Name', 'Description', 'URL', 'LicenseURL', 
            ['LinkedRecords', 'CID']  # Извлекаем дополнительные метаданные
        ],
        meta_prefix='Annotation_',  # Добавляем префикс к столбцам с метаданными
        errors='ignore'  # Игнорируем ошибки, если не удается распаковать вложенные данные
    )

    # Вспомогательная функция для преобразования списка в строку
    def list_to_string(value):
        """Преобразует список в строку, разделяя элементы запятой."""
        if isinstance(value, list):
            return ', '.join(str(item) for item in value)
        return value

    # Вспомогательная функция для извлечения строк с разметкой (StringWithMarkup)
    def extract_string_with_markup(value):
        """Извлекает строковые значения с разметкой из списка и объединяет их в одну строку."""
        if isinstance(value, list):
            strings = [item['String'] for item in value]  
            return ', '.join(strings)  
        return None  

    # Применение вспомогательных функций для преобразования столбцов в DataFrame
    if 'Reference' in annotations_df:
        annotations_df['Reference'] = annotations_df['Reference'].apply(list_to_string)
        
    if 'Annotation_LinkedRecords.CID' in annotations_df:
        annotations_df['Annotation_LinkedRecords.CID'] = annotations_df['Annotation_LinkedRecords.CID'].apply(list_to_string)

    # Преобразуем строковые данные с разметкой в столбец BP_String
    if 'Value.StringWithMarkup' in annotations_df:
        annotations_df['BP_String'] = annotations_df['Value.StringWithMarkup'].apply(extract_string_with_markup)
        annotations_df.drop(columns=['Value.StringWithMarkup'], inplace=True)  # Убираем оригинальный столбец с разметкой

    # Извлекаем числовые значения из строки BP_String (например, температуру)
    annotations_df['Extracted_Value'] = annotations_df['BP_String'].str.extract(r'(-?\d.*)')

    # Функция для извлечения температуры из строки (например, 20°C или -5°F)
    def extract_temperature(value):
        """Извлекает температуру из строки, поддерживает как одиночные значения, так и диапазоны."""
        if pd.isna(value):
            return None, None
        
        # Шаблон для одиночной температуры (например, 20°C, -5°F)
        single_temp_pattern = r'(?<!\d)-?\d+\.?\d*\s*°[CF]'
        # Шаблон для диапазона температур (например, 10-20°C)
        range_temp_pattern = r'\d+\.?\d*\s*-\s*\d+\.?\d*\s*°[CF]'
        
        # Сначала проверяем на диапазон температур
        range_match = re.search(range_temp_pattern, value)
        if range_match:
            # Извлекаем максимальное значение температуры из диапазона
            numbers = re.findall(r'\d+\.?\d*', range_match.group())
            if numbers:
                temp = float(numbers[-1])  # Берем последнее число в диапазоне
                unit = re.search(r'°([CF])', range_match.group()).group()
                return temp, f"°{unit}"
        
        # Если не диапазон, проверяем одиночную температуру
        single_match = re.search(single_temp_pattern, value)
        if single_match:
            temp_part = re.search(r'-?\d+\.?\d*', single_match.group()).group()
            unit = re.search(r'°([CF])', single_match.group()).group()
            return float(temp_part), f"°{unit}"
        
        return None, None

    # Функция для извлечения давления из строки (например, 760 mmHg)
    def extract_pressure(value):
        """Извлекает давление из строки в мм рт. ст. (например, '760 mmHg')."""
        if pd.isna(value):
            return None, None
        match = re.search(r'(\d+)\s*(mm|MM)', value)
        if match:
            return float(match.group(1)), match.group(2)
        return None, None

    # Применяем функции для извлечения температуры и давления
    annotations_df[['Temperature_Value', 'Temperature_Unit']] = annotations_df['Extracted_Value'].apply(extract_temperature).apply(pd.Series)
    annotations_df[['Pressure_Value', 'Pressure_Unit']] = annotations_df['Extracted_Value'].apply(extract_pressure).apply(pd.Series)

    # Функция для преобразования температуры из Фаренгейта в Цельсий
    def fahrenheit_to_celsius(fahrenheit):
        """Преобразует температуру из Фаренгейта в Цельсий."""
        if fahrenheit is not None:
            return (fahrenheit - 32) * 5 / 9
        return None

    # Применяем функцию преобразования температуры, если она в Фаренгейтах
    def convert_temperature(row):
        """Преобразует температуру в Цельсии, если она задана в Фаренгейтах."""
        if row['Temperature_Unit'] == '°F':
            return fahrenheit_to_celsius(row['Temperature_Value'])
        return row['Temperature_Value']

    annotations_df['Temperature_Celsius'] = annotations_df.apply(convert_temperature, axis=1)

    # Функция для преобразования давления из мм рт. ст. в атмосферное давление
    def mmHg_to_atm(mmHg):
        """Преобразует давление из мм рт. ст. в атмосферное (atm)."""
        if mmHg is not None:
            return mmHg / 760  
        return None

    annotations_df['Pressure_Atm'] = annotations_df['Pressure_Value'].apply(mmHg_to_atm)
    
    # Добавляем имя исходного файла для отслеживания данных
    annotations_df['Source_File'] = os.path.basename(file_path)
    
    return annotations_df

In [33]:
# Основной код для обработки всех JSON файлов в указанной директории
def process_all_json_files(directory_path='/home/skvortsovea/practice3/json', file_pattern='NBP*.json'):
    """Обрабатывает все JSON-файлы в указанной директории и возвращает объединённый DataFrame."""
    all_dfs = []
    
    # Находим все подходящие файлы в директории
    json_files = [f for f in os.listdir(directory_path) if f.endswith('.json') or f.endswith('.JSON')  and f.startswith('NBP')]
    
    # Обрабатываем каждый файл
    for json_file in json_files:
        try:
            file_path = os.path.join(directory_path, json_file)
            print(f"Обрабатываю файл: {file_path}")
            df = process_json_file(file_path)  # Обрабатываем текущий файл
            all_dfs.append(df)  # Добавляем DataFrame в общий список
        except Exception as e:
            print(f"Ошибка при обработке файла {json_file}: {str(e)}")
    
    # Если были обработаны файлы, объединяем все DataFrame в один
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        return combined_df
    else:
        print("Не найдено подходящих файлов для обработки")
        return pd.DataFrame()

if __name__ == "__main__":
    # Обрабатываем все JSON файлы в указанной директории
    final_df = process_all_json_files(directory_path='/home/skvortsovea/practice3/json')

    # Если данные были успешно обработаны, сохраняем их в CSV файл
    if not final_df.empty:
        final_df.to_csv('my_nbp_data.csv', index=False)
        print("Данные успешно объединены и сохранены в my_nbp_data.csv")

Обрабатываю файл: /home/skvortsovea/practice3/json/NBP10.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP7.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP5.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP2.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP8.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP1.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP9.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP3.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP4.json
Обрабатываю файл: /home/skvortsovea/practice3/json/NBP6.json
Данные успешно объединены и сохранены в my_nbp_data.csv


In [34]:
# получившийся dataframe:
final_df.shape

(9943, 26)

In [35]:
# Создаем DataFrame с непустыми значениями CID и извлекаем только нужные столбцы
final_df2 = final_df[final_df['Annotation_LinkedRecords.CID'].notna()]
# создадим новый df c перчисленными столбцами
final_df3 = final_df2[['Annotation_LinkedRecords.CID', 'Temperature_Celsius', 'Pressure_Atm']]

# Сортировка по CID, температуре и давлению, установка CID как индекс
final_df3.sort_values(by=['Annotation_LinkedRecords.CID','Temperature_Celsius', 'Pressure_Atm' ])
final_df3.index = final_df3['Annotation_LinkedRecords.CID']
final_df3.index.name = 'CID'

# Оставляем только столбцы температуры и давления
final_df3 = final_df3[['Temperature_Celsius', 'Pressure_Atm']]

# Удаляем строки с пустыми значениями температуры (если есть)
final_df4 = final_df3[final_df3['Temperature_Celsius'].notna()]

# Удаляем дубликаты по индексу, оставляя только первое вхождение для каждого уникального CID
end_df = final_df4[~final_df4.index.duplicated(keep='first')]

# Сохраняем итоговые результаты в файлы
end_df.to_csv('end_df.csv', index=True)  # Сохраняем в CSV с индексом CID
end_df.to_pickle('end_data.pkl')  # Сохраняем в Pickle для дальнейшего использования

In [36]:
end_df

,Temperature_Celsius,Pressure_Atm
CID,,
4748,281.0,0.000000
4033,87.0,0.001316
4927,192.0,0.000000
3100,165.0,0.000000
19649,75.0,1.000000
...,...,...
25797,189.0,0.000000
6450832,259.0,0.000000
5373729,107.0,0.000000


In [37]:
# Выводим размер итогового DataFrame
end_df.shape


(4379, 2)

In [38]:
# Уникальные значения CID сохраняются в set, чтобы избежать дубликатов
unique_cids = set(end_df.index)
# Выводим уникальные CID, чтобы проверить, какие идентификаторы присутствуют в итоговом DataFrame
print(unique_cids)
# Подсчитываем количество уникальных CID в (это количество уникальных молекул в итоговых данных)
count = len(unique_cids)
# Выводим количество уникальных CID
print(count)

{'12210', '7114', '12713', '74530', '7173', '5362595', '15286', '9300', '12252', '11583', '6974', '8079', '86306395', '2796', '11535', '8830', '24268', '10317', '520144', '61290', '8114', '31279', '61660', '19780784', '7242', '7758', '15413', '3086', '5462311', '8379', '5354998', '7394', '17423', '36565', '61319', '15133', '522673', '19602', '5452', '12679', '3342', '12762300', '637563', '14846', '61743', '537615', '11524680', '6954', '243675', '10484', '8054', '91462', '38307', '22251515', '16591', '121677', '24549', '7636', '24516', '164827', '643833', '444294', '14296', '13633097', '12519', '643460', '4754', '23973', '7284', '9477', '23937', '5283363', '8853', '8130', '8038', '71587183', '5362581', '31278', '10795', '11040937', '7981', '2724333', '525358', '15294', '61071', '7041', '11', '5362696', '228588', '22044', '6431144', '7488', '3863468', '18725', '62105', '12581', '11622', '7439', '6430713', '5281553', '285097', '641245', '8094, 135372455', '15037', '19042', '8078', '6476',

In [45]:
def load_data():
    """Загружает данные из Pickle файла и извлекает уникальные CID."""
    # Открытие и загрузка данных из Pickle файла
    with open('end_data.pkl', 'rb') as f:
        data = pickle.load(f)
    
    # Извлечение уникальных CID из индекса данных
    unique_cids = set(data.index)
    
    # Выводим количество уникальных CID, чтобы понять, сколько молекул загружено
    print(f"Загружено {len(unique_cids)} уникальных CID")
    
    # Возвращаем данные и уникальные CID
    return data, unique_cids


def process_sdf_files(data, unique_cids, folder_path):
    """Ищет молекулы в SDF файлах и добавляет данные о температуре и давлении."""
    # Получаем список всех файлов с расширением '.sdf.gz' в указанной директории
    file_list = [f for f in os.listdir(folder_path) if f.endswith('.sdf.gz')]
    
    total_found = 0  # Счётчик найденных молекул
    
    # Указываем имя выходного файла для сохранения молекул с добавленными данными
    output_file = 'Pubchem_with_nbp.sdf.gz'
    
    # Открываем выходной файл для записи данных
    with gzip_open(output_file, 'wt') as gzf, SDFWrite(gzf) as out_file:
        for file_name in file_list:
            file_path = os.path.join(folder_path, file_name)
            print(f"Обработка файла: {file_name}")
            
            try:
                # Открываем текущий SDF файл и начинаем чтение молекул
                with gzip_open(file_path, 'rt') as f, SDFRead(f, buffer_size=100000) as mols_file:
                    for m in mols_file:
                        # Получаем CID текущей молекулы
                        cid = m.meta.get('PUBCHEM_COMPOUND_CID')
                        
                        # Если CID молекулы присутствует в нашем наборе уникальных CID, добавляем данные
                        if cid in unique_cids:
                            try:
                                # Извлекаем свойства из данных, используя CID
                                properties = data.loc[cid]
                                
                                # Добавляем температуру и давление к метаданным молекулы
                                m.meta['Temperature_Celsius'] = properties['Temperature_Celsius']
                                m.meta['Pressure_Atm'] = properties['Pressure_Atm']
                                
                                # Записываем молекулу в выходной файл
                                out_file.write(m)
                                total_found += 1
                                
                                # Каждые 100 найденных молекул выводим информацию
                                if total_found % 100 == 0:
                                    print(f"Найдено молекул: {total_found}")
                                    
                            except Exception as prop_error:
                                # Обрабатываем ошибку, если не удалось добавить свойства для молекулы
                                print(f"Ошибка добавления свойств для CID {cid}: {prop_error}")
            except Exception as file_error:
                # Обрабатываем ошибку при обработке SDF файла
                print(f"Ошибка обработки файла {file_name}: {file_error}")
    
    # Выводим общее количество обработанных молекул
    print(f"Всего найдено и обработано молекул: {total_found}")
    
    # Возвращаем общее количество найденных молекул
    return total_found


if __name__ == "__main__":
    # Указываем путь к директории с SDF файлами
    folder_path = r'/home/share/PubChem_Compounds_DB_19.01.2025'
    
    # Загружаем данные и уникальные CID
    data, unique_cids = load_data()
    
    # Обрабатываем все SDF файлы в указанной директории, добавляем данные и сохраняем в выходной файл
    process_sdf_files(data, unique_cids, folder_path)


Загружено 4379 уникальных CID
Обработка файла: Compound_024500001_025000000.sdf.gz
Обработка файла: Compound_045500001_046000000.sdf.gz
Обработка файла: Compound_028500001_029000000.sdf.gz
Обработка файла: Compound_014000001_014500000.sdf.gz
Обработка файла: Compound_011500001_012000000.sdf.gz
Обработка файла: Compound_027500001_028000000.sdf.gz
Обработка файла: Compound_026000001_026500000.sdf.gz
Обработка файла: Compound_003000001_003500000.sdf.gz
Обработка файла: Compound_034500001_035000000.sdf.gz
Обработка файла: Compound_012500001_013000000.sdf.gz
Обработка файла: Compound_003500001_004000000.sdf.gz
Обработка файла: Compound_018000001_018500000.sdf.gz
Обработка файла: Compound_022500001_023000000.sdf.gz
Обработка файла: Compound_004500001_005000000.sdf.gz
Обработка файла: Compound_043000001_043500000.sdf.gz
Обработка файла: Compound_007000001_007500000.sdf.gz
Обработка файла: Compound_041000001_041500000.sdf.gz
Обработка файла: Compound_004000001_004500000.sdf.gz
Обработка файла:

In [46]:
# Инициализируем счетчики для разных типов молекул
neorg_mol = 0  # Количество молекул неорганики
flag_is = 0  # Количество молекул, у которых очищены изотопы
flag_coord = 0  # Количество молекул, у которых удалены координационные связи
flag_salts = 0  # Количество распавшихся металлических солей
flag_neutr = 0  # Количество молекул, которые были нейтрализованы
flag_components = 0  # Количество молекул с несколькими компонентами
flag_radicals = 0  # Количество молекул, являющихся радикалами
flag_final_molecules = 0  # Количество молекул, прошедших все этапы обработки

# Открываем исходный SDF файл для чтения и файлы для записи
with gzip_open("Pubchem_with_nbp.sdf.gz", 'rt') as f:
    sdf_reader = SDFRead(f)

    # Открываем выходные файлы для записи
    with SDFWrite('nbp_neorg_molecules.sdf') as neorg_file, \
        SDFWrite('nbp_clean_stereo.sdf') as clean_ster, \
        SDFWrite('nbp_stereo.sdf') as ster_file, \
        SDFWrite('pubchem_nbp_final.sdf') as final_file:  # Добавлен новый файл для записи

        # Обрабатываем молекулы из SDF файла
        for i, m in enumerate(sdf_reader):
            # Извлекаем уникальные атомы в молекуле
            unique_atoms = set([a[1].atomic_symbol for a in list(m.atoms())])
            
            # Проверяем, содержат ли молекулы углерод (C) и водород (H)
            if 'C' in unique_atoms and 'H' in unique_atoms:
                # Стандартизируем молекулу (приводим к каноническому виду)
                m.canonicalize(logging=True)
                
                # Если молекула очищена от изотопов, увеличиваем счетчик
                if m.clean_isotopes():
                    flag_is += 1

                # Если молекула очищена от координационных связей, увеличиваем счетчик
                if m.remove_coordinate_bonds():
                    flag_coord += 1

                # Если молекула соль и распадается, увеличиваем счетчик
                if m.split_metal_salts(logging=True):
                    flag_salts += 1

                # Если молекула нейтрализована, увеличиваем счетчик
                if m.neutralize():
                    flag_neutr += 1

                # Если молекула имеет более одного компонента, увеличиваем счетчик
                if m.connected_components_count > 1:
                    flag_components += 1
                else:
                    # Если молекула является радикалом, увеличиваем счетчик радикалов
                    if m.is_radical:
                        flag_radicals += 1
                    else:
                        # Записываем молекулу в соответствующие файлы
                        ster_file.write(m)
                        m.clean_stereo()
                        clean_ster.write(m)
                        final_file.write(m)  # Записываем молекулу в pubchem_nbp_final.sdf
                        flag_final_molecules += 1
            else:
                # Если молекула неорганика, записываем её в отдельный файл
                neorg_file.write(m)
                neorg_mol += 1  # Увеличиваем счетчик молекул, не содержащих углерод и водород

print('Результаты стандартизации:')
# Печатаем результаты обработки молекул
print("=" * 40)
print("Variable".ljust(20) + "Value".rjust(20))
print("=" * 40)

# Печатаем значения счетчиков для каждого типа молекул
print("neorg_mol".ljust(20) + f"{neorg_mol}".rjust(20))
print("flag_is".ljust(20) + f"{flag_is}".rjust(20))
print("flag_coord".ljust(20) + f"{flag_coord}".rjust(20))
print("flag_salts".ljust(20) + f"{flag_salts}".rjust(20))
print("flag_neutr".ljust(20) + f"{flag_neutr}".rjust(20))
print("flag_components".ljust(20) + f"{flag_components}".rjust(20))
print("flag_radicals".ljust(20) + f"{flag_radicals}".rjust(20))
print("flag_final_molecules".ljust(20) + f"{flag_final_molecules}".rjust(20))
print("=" * 40)
print('Итоговый файл, файл с неорганикой и стеореизомеры сохранены в соответствующую папку')


Результаты стандартизации:
Variable                           Value
neorg_mol                            390
flag_is                                0
flag_coord                             0
flag_salts                             0
flag_neutr                             3
flag_components                       58
flag_radicals                          1
flag_final_molecules                3828
Итоговый файл, файл с неорганикой и стеореизомеры сохранены в соответствующую папку
